In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# data path
DATA_PATH = Path(
    r"C:\Users\jchmo\Downloads\dataset_football_matches\big5"
)

In [3]:
# dictionary of the league names
league_names = {
    "E0": "Premier League",
    "D1": "Bundesliga",
    "I1": "Serie A",
    "SP1": "La Liga",
    "F1": "Ligue 1"
}

In [4]:
# load all the data
dataframes = []

for file in sorted(DATA_PATH.glob("*.csv")):

    # eg. E0_2122
    file_name = file.stem

    # split the name to season and league
    league_code, season_code = file_name.split("_")

    # league name
    league = league_names[league_code]

    # 2122 -> 2021/22
    start_year = 2000 + int(season_code[:2])
    end_year = 2000 + int(season_code[2:])

    season = f"{start_year}/{str(end_year)[-2:]}"

    # load CSV
    try:
        df_temp = pd.read_csv(file, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df_temp = pd.read_csv(file, encoding="cp1252")

    # delete empty columns
    df_temp.columns = df_temp.columns.str.strip()

    # add info about league and season
    df_temp["Season"] = season
    df_temp["League"] = league
    df_temp["LeagueCode"] = league_code

    # save the data to dataframe
    dataframes.append(df_temp)

# concatenate all dataframes
df = pd.concat(
    dataframes,
    ignore_index=True,
    sort=False
)

In [5]:
# manage date
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df = df.sort_values("Date").reset_index(drop = True)
print(df["Date"].min())
print(df["Date"].max())
print(df["FTR"].value_counts())


2021-08-06 00:00:00
2026-05-24 00:00:00
FTR
H    3878
A    2769
D    2261
Name: count, dtype: int64


In [6]:
target = "FTR"

metadata_cols = [
    "Div",
    "Date",
    "Time",
    "HomeTeam",
    "AwayTeam",
    "Season",
    "League",
    "LeagueCode",
    "Referee",
]

result_cols = [
    "FTHG",
    "FTAG",
    "FTR",
    "HTHG",
    "HTAG",
    "HTR",
    "HS",
    "AS",
    "HST",
    "AST",
    "HF",
    "AF",
    "HC",
    "AC",
    "HY",
    "AY",
    "HR",
    "AR",
]

In [8]:
import numpy as np

# sort values
df = df.sort_values(["Date", "Time"]).reset_index(drop=True)

# create points per match
df["HomePoints"] = np.where(
    df["FTR"] == "H", 3,
    np.where(df["FTR"] == "D", 1, 0)
)

df["AwayPoints"] = np.where(
    df["FTR"] == "A", 3,
    np.where(df["FTR"] == "D", 1, 0)
)

# create home and away history
home_history = pd.DataFrame({
    "Date": df["Date"],
    "Season": df["Season"],
    "League": df["League"],
    "Team": df["HomeTeam"],
    "Opponent": df["AwayTeam"],
    "HomeAway": "H",

    "Points": df["HomePoints"],
    "GoalsFor": df["FTHG"],
    "GoalsAgainst": df["FTAG"],

    "Shots": df["HS"],
    "ShotsOnTarget": df["HST"],
    "Corners": df["HC"],
    "Fouls": df["HF"],
    "YellowCards": df["HY"],
    "RedCards": df["HR"],
})


away_history = pd.DataFrame({
    "Date": df["Date"],
    "Season": df["Season"],
    "League": df["League"],
    "Team": df["AwayTeam"],
    "Opponent": df["HomeTeam"],
    "HomeAway": "A",

    "Points": df["AwayPoints"],
    "GoalsFor": df["FTAG"],
    "GoalsAgainst": df["FTHG"],

    "Shots": df["AS"],
    "ShotsOnTarget": df["AST"],
    "Corners": df["AC"],
    "Fouls": df["AF"],
    "YellowCards": df["AY"],
    "RedCards": df["AR"],
})

team_history = pd.concat(
    [home_history, away_history],
    ignore_index=True
)

team_history = team_history.sort_values(
    ["Team", "Date"]
).reset_index(drop=True)

In [9]:
team_history["Last5Points"] = (
    team_history
    .groupby("Team")["Points"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
)

In [10]:
grouped = team_history.groupby("Team")

team_history["Last5Points"] = (
    grouped["Points"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

team_history["Last5GoalsFor"] = (
    grouped["GoalsFor"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

team_history["Last5GoalsAgainst"] = (
    grouped["GoalsAgainst"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

team_history["Last5Shots"] = (
    grouped["Shots"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

team_history["Last5ShotsOnTarget"] = (
    grouped["ShotsOnTarget"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

team_history["Last5Corners"] = (
    grouped["Corners"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

In [20]:
print(team_history["Last5Points"])

0        NaN
1        0.0
2        1.0
3        1.0
4        1.0
        ... 
17811    7.0
17812    4.0
17813    2.0
17814    1.0
17815    2.0
Name: Last5Points, Length: 17816, dtype: float64
